# XGBoost for preference learning

In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Resolve project root whether notebook runs from workspace root or notebooks/
cwd = Path.cwd()
project_root = cwd if (cwd / "data").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.metrics import evaluate_model

## Load data
We use `hospital_overall_rating` as a binary target variable:
- `0` for bad hospital (`rating < 4`)
- `1` for good hospital (`rating >= 4`)

Metrics are computed with `src.metrics.evaluate_model` and rounded to 4 decimals.

In [2]:
data_path = project_root / "data" / "hospital_data.csv"
df = pd.read_csv(data_path)

# Keep names/ids for reporting and explainability artifacts
meta_cols = ["facility_id", "facility_name"]

cost_cols = [
    "mort_ami",
    "comp_hip_knee",
    "readmission_hf",
    "spending",
    "count_of_readm_measures_worse",
]
gain_cols = ["count_of_safety_measures_better"]

model_df = df.copy()

# Binarized target: 1 for good hospital, 0 otherwise
model_df["target"] = (model_df["hospital_overall_rating"] >= 4).astype(int)

feature_cols = cost_cols + gain_cols
X = model_df[feature_cols]
y = model_df["target"]

print("Rows:", len(model_df))
print("Class distribution:")
print(y.value_counts(normalize=True).rename("share").round(4))

Rows: 500
Class distribution:
target
0    0.602
1    0.398
Name: share, dtype: float64


## Data splitting

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (400, 6)
Test shape: (100, 6)


## Initialize and train XGBoost with monotone constraints
Cost criteria use `-1`, gain criteria use `+1`.

In [4]:
constraints = tuple([-1] * len(cost_cols) + [1] * len(gain_cols))

model = xgb.XGBClassifier(
    random_state=42,
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    monotone_constraints=constraints,
    n_jobs=-1,
)

model.fit(X_train, y_train)
print("Model trained with constraints:", constraints)

Model trained with constraints: (-1, -1, -1, -1, -1, 1)


## Predict and evaluate
Use class predictions for Accuracy/F1 and probabilities for AUC.

In [5]:
y_pred_test = model.predict(X_test)
y_prob_test = model.predict_proba(X_test)[:, 1]

metrics = evaluate_model(y_test, y_pred_test, y_prob_test)
print("Test metrics (rounded to 4 decimals):")
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

Test metrics (rounded to 4 decimals):
accuracy: 0.7900
f1: 0.7200
auc: 0.8533


## Save model
Persist the trained model for SHAP/PDP/DALEX analysis in later notebooks.

In [6]:
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "xgboost_hospital_monotonic.joblib"
joblib.dump(
    {
        "model": model,
        "feature_cols": feature_cols,
        "cost_cols": cost_cols,
        "gain_cols": gain_cols,
        "target_rule": "hospital_overall_rating >= 4 -> 1 else 0",
    },
    model_path,
)

print(f"Saved model bundle to: {model_path}")

Saved model bundle to: /home/marek/put/DA/mcda/models/xgboost_hospital_monotonic.joblib
